# Cvičenie 6 - Bayesov klasifikátor

<object data="https://miroslava.matejova.website.tuke.sk/IE/IE/cvicenie6.pdf" type="application/pdf" width="700px" height="700px">
    <embed src="https://miroslava.matejova.website.tuke.sk/IE/IE/cvicenie6.pdf">
        <p> Cvičenie v PDF dostupné: <a href="https://miroslava.matejova.website.tuke.sk/IE/IE/cvicenie6.pdf">TU!</a></p>
    </embed>
</object>

Ako príklad si uvedieme postup pre klasifikáciu do dvoch tried (`iris setosa` a `iris versicolor`).
Načítame si dáta a vytvoríme si podmnožiny pre jednotlivé triedy:

In [17]:
data(iris)
setosa = iris[iris$Species == "setosa",]
setosa
versicolor = iris[iris$Species == "versicolor",]
versicolor

,Sepal.Length,Sepal.Width,Petal.Length,Petal.Width,Species
,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
1,5.1,3.5,1.4,0.2,setosa
2,4.9,3.0,1.4,0.2,setosa
3,4.7,3.2,1.3,0.2,setosa
4,4.6,3.1,1.5,0.2,setosa
5,5.0,3.6,1.4,0.2,setosa
6,5.4,3.9,1.7,0.4,setosa
7,4.6,3.4,1.4,0.3,setosa
8,5.0,3.4,1.5,0.2,setosa
9,4.4,2.9,1.4,0.2,setosa


,Sepal.Length,Sepal.Width,Petal.Length,Petal.Width,Species
,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
51,7.0,3.2,4.7,1.4,versicolor
52,6.4,3.2,4.5,1.5,versicolor
53,6.9,3.1,4.9,1.5,versicolor
54,5.5,2.3,4.0,1.3,versicolor
55,6.5,2.8,4.6,1.5,versicolor
56,5.7,2.8,4.5,1.3,versicolor
57,6.3,3.3,4.7,1.6,versicolor
58,4.9,2.4,3.3,1.0,versicolor
59,6.6,2.9,4.6,1.3,versicolor


In [18]:
setosa = as.matrix(setosa[1:4])
setosa
versicolor = as.matrix(versicolor[1:4])
versicolor

,Sepal.Length,Sepal.Width,Petal.Length,Petal.Width
1,5.1,3.5,1.4,0.2
2,4.9,3.0,1.4,0.2
3,4.7,3.2,1.3,0.2
4,4.6,3.1,1.5,0.2
5,5.0,3.6,1.4,0.2
6,5.4,3.9,1.7,0.4
7,4.6,3.4,1.4,0.3
8,5.0,3.4,1.5,0.2
9,4.4,2.9,1.4,0.2
10,4.9,3.1,1.5,0.1


,Sepal.Length,Sepal.Width,Petal.Length,Petal.Width
51,7.0,3.2,4.7,1.4
52,6.4,3.2,4.5,1.5
53,6.9,3.1,4.9,1.5
54,5.5,2.3,4.0,1.3
55,6.5,2.8,4.6,1.5
56,5.7,2.8,4.5,1.3
57,6.3,3.3,4.7,1.6
58,4.9,2.4,3.3,1.0
59,6.6,2.9,4.6,1.3
60,5.2,2.7,3.9,1.4


Vypočítame si `P(Y=y)` pre obe triedy:

In [19]:
p_setosa = nrow(setosa)/(nrow(setosa)+nrow(versicolor))
p_setosa
p_versicolor = nrow(versicolor)/(nrow(setosa)+nrow(versicolor))
p_versicolor

[1] 0.5

[1] 0.5

Vypočítame si kovariančné matice a vektory stredov pre jednotlivé triedy:

In [20]:
C_setosa = cov(setosa)
C_setosa
C_versicolor = cov(versicolor)
C_versicolor

,Sepal.Length,Sepal.Width,Petal.Length,Petal.Width
Sepal.Length,0.12424898,0.099216327,0.016355102,0.010330612
Sepal.Width,0.09921633,0.143689796,0.011697959,0.009297959
Petal.Length,0.01635510,0.011697959,0.030159184,0.006069388
Petal.Width,0.01033061,0.009297959,0.006069388,0.011106122


,Sepal.Length,Sepal.Width,Petal.Length,Petal.Width
Sepal.Length,0.26643265,0.08518367,0.18289796,0.05577959
Sepal.Width,0.08518367,0.09846939,0.08265306,0.04120408
Petal.Length,0.18289796,0.08265306,0.22081633,0.07310204
Petal.Width,0.05577959,0.04120408,0.07310204,0.03910612


In [21]:
mean_setosa = colMeans(setosa)
mean_setosa
mean_versicolor = colMeans(versicolor)
mean_versicolor

Sepal.Length  Sepal.Width Petal.Length  Petal.Width 
       5.006        3.428        1.462        0.246

Sepal.Length  Sepal.Width Petal.Length  Petal.Width 
       5.936        2.770        4.260        1.326

Pre nový príklad s hodnotami `x = (5.7, 2.8, 4.1, 1.3)` vypočítame pravdepodobnosť `𝑃(𝑋 =
𝒙|𝑌 = 𝑐)` dosadením **kovariančnej matice** a **vektora stredov pre jednotlivé triedy**. Pre
viachodnotové normálne rozdelenie môžeme pravdepodobnosť `𝑃(𝑋 = 𝒙|𝑌 = 𝑐)` vypočítať
pomocou funkcie `dvnorm` z balíka `emdbook`:

In [22]:
install.packages("emdbook")

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [23]:
library(emdbook)

In [24]:
x = c(5.7, 2.8, 4.1, 1.3) # tento priklad chceme klasifikovat

In [25]:
p_x_setosa = dmvnorm(x, mean_setosa, C_setosa)
p_x_versicolor = dmvnorm(x, mean_versicolor, C_versicolor)

p_x_setosa
p_x_versicolor

[1] 2.227119e-62

[1] 4.817479

In [26]:
p_setosa * p_x_setosa
p_versicolor * p_x_versicolor

[1] 1.11356e-62

[1] 2.408739

Keďže hodnota pre celkový výraz `𝑃(𝑌 = 𝑐)𝑃(𝑋 = 𝒙|𝑌 = 𝑐)` je oveľa väčšia pre triedu
`versicolor`, príklad s hodnotami `x = (5.7, 2.8, 4.1, 1.3)` by sme zaradili do triedy `versicolor`.

Kovariačná matica musí byť **symetrická a pozitívne definitná**. To, či je matica pozitívne definitná môžeme zistiť pomocou funkcie `is.positive.definite()` z knižnice `matrixcalc`. Viac k pozitívne definitným maticiam nájdete [tu](https://www.math.utah.edu/~zwick/Classes/Fall2012_2270/Lectures/Lecture33_with_Examples.pdf).

In [ ]:
install.packages("matrixcalc")
library(matrixcalc)
A <- matrix( c( 2, -1, 0, -1, 2, -1, 0, -1, 2 ), nrow=3, byrow=TRUE )
is.positive.definite(A)

## Úlohy

1. Načítajte si dáta iris a vypočítajte parametre Bayesovho klasifikátora pre všetky tri
triedy `setosa`, `versicolor` a `virginica`.

In [1]:
data(iris)
head(iris)

,Sepal.Length,Sepal.Width,Petal.Length,Petal.Width,Species
,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
1,5.1,3.5,1.4,0.2,setosa
2,4.9,3.0,1.4,0.2,setosa
3,4.7,3.2,1.3,0.2,setosa
4,4.6,3.1,1.5,0.2,setosa
5,5.0,3.6,1.4,0.2,setosa
6,5.4,3.9,1.7,0.4,setosa


In [2]:
setosa = iris[iris$Species=="setosa", ]
versicolor = iris[iris$Species=="versicolor", ]
virginica = iris[iris$Species=="virginica", ]

In [3]:
head(setosa)

,Sepal.Length,Sepal.Width,Petal.Length,Petal.Width,Species
,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
1,5.1,3.5,1.4,0.2,setosa
2,4.9,3.0,1.4,0.2,setosa
3,4.7,3.2,1.3,0.2,setosa
4,4.6,3.1,1.5,0.2,setosa
5,5.0,3.6,1.4,0.2,setosa
6,5.4,3.9,1.7,0.4,setosa


In [7]:
setosa = as.matrix(setosa[1:4])
versicolor = as.matrix(versicolor[1:4])
virginica = as.matrix(virginica[1:4])

In [27]:
N = nrow(setosa)+nrow(versicolor)+nrow(virginica)
ps = nrow(setosa)/N
pve = nrow(versicolor)/N
pvi = nrow(virginica)/N

N
ps
pve
pvi

[1] 150

[1] 0.3333333

[1] 0.3333333

[1] 0.3333333

In [28]:
cs = cov(setosa)
cve = cov(versicolor)
cvi = cov(virginica)
cs
cve
cvi

,Sepal.Length,Sepal.Width,Petal.Length,Petal.Width
Sepal.Length,0.12424898,0.099216327,0.016355102,0.010330612
Sepal.Width,0.09921633,0.143689796,0.011697959,0.009297959
Petal.Length,0.01635510,0.011697959,0.030159184,0.006069388
Petal.Width,0.01033061,0.009297959,0.006069388,0.011106122


,Sepal.Length,Sepal.Width,Petal.Length,Petal.Width
Sepal.Length,0.26643265,0.08518367,0.18289796,0.05577959
Sepal.Width,0.08518367,0.09846939,0.08265306,0.04120408
Petal.Length,0.18289796,0.08265306,0.22081633,0.07310204
Petal.Width,0.05577959,0.04120408,0.07310204,0.03910612


,Sepal.Length,Sepal.Width,Petal.Length,Petal.Width
Sepal.Length,0.40434286,0.09376327,0.30328980,0.04909388
Sepal.Width,0.09376327,0.10400408,0.07137959,0.04762857
Petal.Length,0.30328980,0.07137959,0.30458776,0.04882449
Petal.Width,0.04909388,0.04762857,0.04882449,0.07543265


In [29]:
ms = colMeans(setosa)
mve= colMeans(versicolor)
mvi= colMeans(virginica)

ms
mve
mvi

Sepal.Length  Sepal.Width Petal.Length  Petal.Width 
       5.006        3.428        1.462        0.246

Sepal.Length  Sepal.Width Petal.Length  Petal.Width 
       5.936        2.770        4.260        1.326

Sepal.Length  Sepal.Width Petal.Length  Petal.Width 
       6.588        2.974        5.552        2.026

In [11]:
install.packages("emdbook")
library(emdbook)

also installing the dependencies ‘bdsmatrix’, ‘mvtnorm’, ‘coda’, ‘bbmle’


Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [30]:
x = c(5.7, 2.8, 4.1, 1.3)
s = dmvnorm(x, ms, cs)
ve = dmvnorm(x, mve, cve)
vi = dmvnorm(x, mvi, cvi)

In [31]:
s*ps
ve*pve
vi*pvi

[1] 7.423731e-63

[1] 1.605826

[1] 0.0003859916

2. Klasifikujte všetky príklady, tzn. vypočítajte pre každý príklad z množiny `iris`
pravdepodobnosti pre všetky tri triedy a určite do ktorej triedy by príklad patril.
Vypočítajte celkovú presnosť klasifikácie (počet správne klasifikovaných príkladov /
počet všetkých príkladov)

3. Nainštalujte si balík `MASS` a pomocou funkcie `mvrnorm(n, means, C)` si vygenerujte `N
= 500` príkladov s viachodnotovým normálnym rozdelením so stredmi `means = c(2,3)`
a kovariančnou maticou `C = matrix(c(9,6,6,16),2,2)`. Zobrazte vygenerované dáta na
grafe. Postupne meňte hodnoty matice C, vygenerujte si nové dáta a pozorujte ako sa
zmenia na grafe:

    * a. Nastavte hodnoty mimo diagonály na 0.
    * b. Nastavte hodnoty na diagonále na 10,10 a 2,2.
    * c. Nastavte hodnoty mimo diagonály na -3

In [32]:
install.packages("MASS")
library(MASS)

N=500
mean = c(2,3)
C = matrix(c(9,6,6,16),2,2)

data = mvrnorm(N, mean, C)

Warning message:
“package ‘MASS’ is not available for this version of R
‘MASS’ version 7.3-61 is in the repositories but depends on R (>= 4.4.0)
‘MASS’ version 7.3-61 is in the repositories but depends on R (>= 4.5)

A version of this package for your version of R might be available elsewhere,
see the ideas at
https://cran.r-project.org/doc/manuals/r-patched/R-admin.html#Installing-packages”


4. Pomocou funkcie mvrnorm si vygenerujte dve dátové množiny o veľkosti `100` a `200`
príkladov pre triedy so stredmi `(10,20)` a `(20,30)` a kovariančnými maticami `[(6,2), (2,8)] a [(6,-3), (-3,6)]`. Vypočítajte parametre Bayesovho klasifikátora a klasifikujte
príklad s hodnotami `(15, 25)`.